[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S11_pandas_agrupar_resumir.ipynb)

# Sesión 11 · pandas: agrupar y resumir

**Módulo 3: Pandas** · ⏱️ Duración estimada: 60 minutos

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Resumir por grupos con `groupby`, con una o varias funciones (`agg`).
2. Contar categorías y proporciones con `value_counts` y cruzar dos variables con `crosstab`.
3. Construir tablas dinámicas con `pivot_table`.
4. Calcular tasas de conversión por segmento y ordenar resultados con `sort_values` y `nlargest`.

## 📋 Qué debes saber antes
Sesiones 9 y 10: seleccionar, filtrar y limpiar DataFrames.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- En cada ✍️ **Tu turno** escribe tu código debajo de `# Tu código aquí`.
- Después ejecuta la celda ✅ **Verificar**. Si aparece ❌, lee el motivo, corrige y vuelve a verificar.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.
- Deja los resultados de `groupby` con los grupos como índice (sin `reset_index`), salvo que se pida otra cosa.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Crea tres DataFrames (`ventas`, `contactos` y `movs`) y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Prepara los datos de la sesión y las funciones que revisan tus respuestas.
import copy
import hashlib
import math
import statistics
from collections import defaultdict

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- Datos de práctica: ventas de tiendas ----------
_TIENDAS = ["Barranco", "Lince", "Miraflores", "San Isidro", "Surco"]
_CATALOGO = [("polo básico", "polo", 39.9), ("polo piqué", "polo", 59.9), ("jean slim", "jean", 129.9),
             ("jean mom", "jean", 149.9), ("casaca denim", "casaca", 189.9), ("gorra", "accesorio", 25.0),
             ("medias", "accesorio", 12.9)]
_filas = []
for _i in range(200):
    _p = _CATALOGO[int(rng.integers(0, len(_CATALOGO)))]
    _canal = str(rng.choice(["app", "tienda", "web"], p=[0.3, 0.5, 0.2]))
    if _p[1] == "accesorio" and _canal == "web":
        _canal = "tienda"                                   # los accesorios no se venden por web
    _u = int(rng.integers(1, 7))
    _filas.append([f"2026-09-{int(rng.integers(1, 31)):02d}", str(rng.choice(_TIENDAS, p=[0.15, 0.17, 0.25, 0.2, 0.23])),
                   _p[0], _p[1], _canal, _u, _p[2], round(_u * _p[2], 2)])
ventas = pd.DataFrame(_filas, columns=["fecha", "tienda", "producto", "categoria", "canal", "unidades", "precio", "importe"])

# ---------- Datos de práctica: campaña comercial ----------
_base = {"email": 0.08, "sms": 0.05, "llamada": 0.18}
_ajuste = {"joven": 0.04, "adulto": 0.0, "senior": -0.03}
_c = []
for _i in range(300):
    _seg = str(rng.choice(list(_ajuste)))
    _can = str(rng.choice(list(_base)))
    _conv = int(rng.random() < _base[_can] + _ajuste[_seg])
    _c.append([f"K{_i + 1:03d}", _seg, _can, _conv, round(float(rng.uniform(80, 900)), 2) if _conv else 0.0])
contactos = pd.DataFrame(_c, columns=["contacto", "segmento", "canal_contacto", "convirtio", "monto_compra"])

# ---------- Datos de práctica: movimientos bancarios ----------
_SEG_CLIENTE = {f"C{k:02d}": str(rng.choice(["clásico", "preferente", "premium"], p=[0.5, 0.3, 0.2])) for k in range(1, 21)}
_m = []
for _i in range(180):
    _cl = f"C{int(rng.integers(1, 21)):02d}"
    _tipo = str(rng.choice(["deposito", "retiro", "pago", "transferencia"], p=[0.25, 0.35, 0.25, 0.15]))
    _canal = str(rng.choice(["app", "agencia", "cajero"])) if _tipo == "retiro" else str(rng.choice(["app", "agencia"]))
    _monto = rng.uniform(200, 4000) if _tipo == "deposito" else -rng.uniform(20, 1500)
    _m.append([_cl, _SEG_CLIENTE[_cl], _tipo, _canal, round(float(_monto), 2), int(rng.integers(1, 7))])
_m += [["C21", "premium", "deposito", "app", 5200.0, 2], ["C21", "premium", "deposito", "agencia", 3100.0, 5]]   # cliente sin egresos
movs = pd.DataFrame(_m, columns=["cliente", "segmento", "tipo", "canal", "monto", "mes"])

_D = copy.deepcopy({"ventas": ventas, "contactos": contactos, "movs": movs})
# Versiones en listas de Python: los verificadores recalculan con bucles y diccionarios, sin pandas.
_R = {k: [dict(zip(v.columns, map(lambda x: x.item() if hasattr(x, "item") else x, fila))) for fila in v.itertuples(index=False)]
      for k, v in _D.items()}

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")


def _agrupar(filas, clave, valor, fun, filtro=lambda f: True):
    grupos = defaultdict(list)
    for f in filas:
        if filtro(f):
            k = tuple(f[c] for c in clave) if isinstance(clave, (list, tuple)) else f[clave]
            grupos[k].append(f[valor])
    return {k: fun(v) for k, v in sorted(grupos.items())}


def _media(xs):
    return math.fsum(xs) / len(xs)


def _ser_dict(r, nombre, d, pista, tol=1e-6):
    _ser(r, nombre, list(d.values()), pista, indice=list(d.keys()), tol=tol)


def _tabla(filas, fila, col, valor=None, fun=len, fill=None, normalizar=False):
    """Tabla cruzada de referencia: dict fila -> dict col -> valor."""
    filas_k = sorted({f[fila] for f in filas})
    cols_k = sorted({f[col] for f in filas})
    salida = {}
    for a in filas_k:
        fila_d = {}
        for b in cols_k:
            vals = [f[valor] if valor else 1 for f in filas if f[fila] == a and f[col] == b]
            fila_d[b] = fun(vals) if vals else fill
        if normalizar:
            tot = math.fsum(fila_d.values())
            fila_d = {b: x / tot for b, x in fila_d.items()}
        salida[a] = fila_d
    return filas_k, cols_k, salida


def _df_tabla(r, nombre, ref, pista, tol=1e-6):
    filas_k, cols_k, d = ref
    _df(r, nombre, [str(c) for c in cols_k], [[d[a][b] for b in cols_k] for a in filas_k], pista, indice=filas_k, tol=tol)


def check_ejercicio_1():
    r = _Revision("Ejercicio 1 · Parte A")
    v = _R["ventas"]
    _ser_dict(r, "total_por_tienda", _agrupar(v, "tienda", "importe", math.fsum), "suma el importe de cada tienda")
    _ser_dict(r, "unidades_por_categoria", _agrupar(v, "categoria", "unidades", sum), "suma las unidades de cada categoría")
    _ser_dict(r, "ticket_medio_canal", _agrupar(v, "canal", "importe", lambda xs: round(_media(xs), 2)),
              "promedio del importe por canal, con 2 decimales", tol=0.0051)
    _ser_dict(r, "ventas_tienda_canal", _agrupar(v, ["tienda", "canal"], "importe", math.fsum),
              "agrupa por tienda y canal, en ese orden, y suma el importe")
    _sin_cambios_df(r, "ventas")
    r.fin()
    r = _Revision("Ejercicio 1 · Parte B")
    r.predicciones({
        "pred_grupo_a": "7a20ad6fbd72c40ec9ce42fcce75b2afd01e02ef19725a1956c8b5a8b66eeefc",
        "pred_n_grupos": "b4d4f68c2268549cada66d24ae3a1902d4152a98ad614bbf891edf235ef99218",
    })
    r.fin()


def check_ejercicio_2():
    r = _Revision("Ejercicio 2")
    v = _R["ventas"]
    tiendas = sorted({f["tienda"] for f in v})
    filas = []
    for t in tiendas:
        imp = [f["importe"] for f in v if f["tienda"] == t]
        filas.append([math.fsum(imp), _media(imp), len(imp), max(f["unidades"] for f in v if f["tienda"] == t)])
    _df(r, "resumen_tienda", ["total", "promedio", "n_ventas", "max_unidades"], filas,
        "revisa los nombres de las columnas, qué columna resume cada una y con qué función", indice=tiendas)
    cats = sorted({f["categoria"] for f in v})
    filas = [[math.fsum(x), _media(x), len(x)] for x in ([f["importe"] for f in v if f["categoria"] == c] for c in cats)]
    _df(r, "resumen_categoria", ["sum", "mean", "count"], filas, "suma, promedio y conteo del importe por categoría", indice=cats)
    r.fin()


def check_ejercicio_3():
    r = _Revision("Ejercicio 3 · Parte A")
    v = _R["ventas"]
    conteo = _agrupar(v, "canal", "canal", len)
    for nombre, esperado, tol in (("conteo_canal", conteo, 0),
                                  ("prop_canal", {k: round(x / len(v), 3) for k, x in conteo.items()}, 0.00051)):
        s = r.var(nombre)
        if s is _FALTA:
            continue
        if not isinstance(s, pd.Series):
            r.mal(f"`{nombre}` es de tipo {type(s).__name__} y se esperaba una Series de pandas.")
        elif sorted(map(str, s.index)) != sorted(esperado) or any(abs(float(s[k]) - esperado[k]) > tol + 1e-9 for k in esperado):
            r.mal(f"`{nombre}` no tiene los valores esperados; " + ("cuenta cuántas ventas hay de cada canal." if tol == 0 else "usa `normalize=True` y redondea a 3 decimales."))
        elif any(a < b for a, b in zip(s.tolist(), s.tolist()[1:])):
            r.mal(f"`{nombre}` debería estar ordenado de mayor a menor, como lo deja `value_counts`.")
        else:
            r.ok(f"`{nombre}` es correcto.")
    _df_tabla(r, "tabla_cruzada", _tabla(v, "tienda", "canal", fill=0), "cuenta las ventas de cada tienda (filas) en cada canal (columnas)")
    _df_tabla(r, "mezcla_canal", _tabla(v, "tienda", "canal", fill=0, normalizar=True),
              "la proporción de cada canal dentro de cada tienda (cada fila suma 1), con 3 decimales", tol=0.00051)
    r.fin()
    r = _Revision("Ejercicio 3 · Parte B")
    r.predicciones({
        "pred_suma_normalize": "b399e5afcd7822a4082968fd88be832ae2163fda3da62ed3f6d9402bfba8dd1c",
    })
    r.fin()


def check_ejercicio_4():
    r = _Revision("Ejercicio 4 · Parte A")
    v = _R["ventas"]
    _df_tabla(r, "pivote", _tabla(v, "tienda", "categoria", "importe", math.fsum, fill=0),
              "suma del importe con tiendas en las filas, categorías en las columnas y 0 donde no hay ventas")
    _df_tabla(r, "pivote_unid", _tabla(v, "categoria", "canal", "unidades", lambda xs: round(_media(xs), 2)),
              "promedio de unidades con categorías en las filas y canales en las columnas, con 2 decimales (sin rellenar)", tol=0.0051)
    r.fin()
    r = _Revision("Ejercicio 4 · Parte B")
    r.predicciones({
        "pred_sin_fill": "7cd2a33d8476047b3755295f8b4747a3baea572e22f0e24d21505be7bc3c9844",
    })
    r.fin()


def check_ejercicio_5():
    r = _Revision("Ejercicio 5")
    c, v = _R["contactos"], _R["ventas"]
    tasa = _agrupar(c, "segmento", "convirtio", lambda xs: round(_media(xs), 3))
    _ser_dict(r, "tasa_segmento", tasa, "la proporción de contactos que compraron en cada segmento, con 3 decimales", tol=0.00051)
    r.valor("mejor_segmento", max(tasa, key=tasa.get), None, "debería ser la etiqueta del segmento con mayor tasa",
            igual=lambda a, b: isinstance(a, str) and a == b)
    por_canal = _agrupar(c, "canal_contacto", "convirtio", _media)
    orden = sorted(por_canal, key=lambda k: -por_canal[k])
    _ser(r, "tasa_canal_ordenada", [por_canal[k] for k in orden], "tasa por canal ordenada de mayor a menor", indice=orden)
    tot = _agrupar(v, "producto", "importe", math.fsum)
    top = sorted(tot, key=lambda k: -tot[k])[:3]
    _ser(r, "top3_productos", [tot[k] for k in top], "los 3 productos con mayor importe total, de mayor a menor", indice=top)
    x = r.var("ventas_ordenadas")
    if x is not _FALTA:
        if not isinstance(x, pd.DataFrame) or sorted(x.index.tolist()) != list(range(len(v))):
            r.mal("`ventas_ordenadas` debería tener las mismas filas que `ventas`, solo que en otro orden.")
        else:
            pares = list(zip(x["tienda"].tolist(), x["importe"].tolist()))
            if any(a[0] > b[0] or (a[0] == b[0] and a[1] < b[1]) for a, b in zip(pares, pares[1:])):
                r.mal("`ventas_ordenadas` debería ir por tienda de la A a la Z y, dentro de cada tienda, de mayor a menor importe.")
            else:
                r.ok("`ventas_ordenadas` es correcto.")
    r.fin()


def check_reto():
    r = _Revision("Reto final")
    m = _R["movs"]
    _ser_dict(r, "egresos_cliente", _agrupar(m, "cliente", "monto", lambda xs: round(math.fsum(xs), 2), filtro=lambda f: f["monto"] < 0),
              "filtra primero los montos negativos y luego suma por cliente, con 2 decimales", tol=0.0051)
    segs = sorted({f["segmento"] for f in m})
    filas = []
    for s in segs:
        sub = [f for f in m if f["segmento"] == s]
        filas.append([len({f["cliente"] for f in sub}), len(sub), round(_media([f["monto"] for f in sub]), 2)])
    _df(r, "resumen_segmento", ["n_clientes", "n_movs", "monto_medio"], filas,
        "clientes distintos, cantidad de movimientos y monto promedio por segmento, con 2 decimales", indice=segs, tol=0.0051)
    fk, ck, d = _tabla(m, "segmento", "canal", fill=0, normalizar=True)
    _df_tabla(r, "uso_canal", (fk, ck, {a: {b: round(x, 3) for b, x in fila.items()} for a, fila in d.items()}),
              "proporción de cada canal dentro de cada segmento, con 3 decimales", tol=0.00051)
    fk, ck, d = _tabla(m, "mes", "tipo", "monto", math.fsum, fill=0)
    _df_tabla(r, "mes_tipo", (fk, ck, d), "suma del monto con meses en las filas y tipos en las columnas, con 0 donde no hay datos", tol=0.0051)
    dep = _agrupar(m, "cliente", "monto", math.fsum, filtro=lambda f: f["tipo"] == "deposito")
    top = sorted(dep, key=lambda k: -dep[k])[:5]
    _ser(r, "top5_depositantes", [round(dep[k], 2) for k in top], "los 5 clientes con más dinero depositado, de mayor a menor", indice=top, tol=0.0051)
    pct = _agrupar(m, "segmento", "canal", lambda xs: round(len([x for x in xs if x == "app"]) * 100 / len(xs), 1))
    _ser_dict(r, "pct_app_segmento", pct, "porcentaje de movimientos hechos por app en cada segmento, con 1 decimal", tol=0.051)
    _sin_cambios_df(r, "movs")
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    v = _R["ventas"]
    tot = _agrupar(v, "tienda", "importe", math.fsum)
    _ser(r, "participacion", [round(f["importe"] / tot[f["tienda"]], 4) for f in v],
         "cada importe entre el total de su tienda, con 4 decimales, en el orden original de `ventas`", tol=0.00006)
    rangos = []
    for f in v:
        rangos.append(1 + len([g for g in v if g["tienda"] == f["tienda"] and g["importe"] > f["importe"]]))
    _ser(r, "rank_en_tienda", rangos, "la posición de cada venta dentro de su tienda: 1 es el mayor importe y los empates comparten el menor puesto")
    r.fin()


print("✅ Setup listo. Datos generados y verificadores cargados.")

### 📦 Tus datos de hoy
- `ventas`: 200 ventas de 5 tiendas en setiembre.
- `contactos`: 300 personas contactadas en una campaña; `convirtio` vale 1 si compraron y 0 si no.
- `movs`: 182 movimientos bancarios de clientes de tres segmentos.

Los valores se generan con una semilla fija, así que siempre salen iguales.

In [ ]:
print(ventas.head(), "\n")
print(contactos.head(), "\n")
print(movs.head())

---
## 1. `groupby`: dividir, aplicar, combinar

### 📘 Concepto
`groupby` responde preguntas del tipo "¿cuánto vendió **cada** tienda?". Hace tres cosas:
1. **Divide** las filas en grupos según una columna.
2. **Aplica** una función a cada grupo (`sum`, `mean`, `count`, `max`...).
3. **Combina** los resultados en una Series o un DataFrame cuyo índice son los grupos, ordenados.

```python
df.groupby("columna_grupo")["columna_valor"].funcion()
```

Con una lista de columnas (`groupby(["tienda", "canal"])`) agrupas por combinaciones y el resultado tiene un índice de dos niveles.

In [ ]:
pedidos_ej = pd.DataFrame({"vendedor": ["Ana", "Luis", "Ana", "Rosa", "Luis"],
                           "canal": ["app", "tienda", "tienda", "app", "app"],
                           "monto": [120.0, 80.0, 45.5, 300.0, 60.0]})
print(pedidos_ej.groupby("vendedor")["monto"].sum())
print(pedidos_ej.groupby("canal")["monto"].mean().round(1))
print(pedidos_ej.groupby(["vendedor", "canal"])["monto"].sum())

### ✍️ Tu turno · Ejercicio 1: totales por grupo
**Parte A.** Con `ventas`:
1. `total_por_tienda`: el importe total de cada tienda.
2. `unidades_por_categoria`: las unidades vendidas de cada categoría.
3. `ticket_medio_canal`: el importe promedio por canal, redondeado a 2 decimales.
4. `ventas_tienda_canal`: el importe total por tienda y canal (en ese orden).

**Parte B.** Predice **sin ejecutar**, con `d = pd.DataFrame({"g": ["a", "b", "a"], "x": [1, 2, 3]})`:

| Variable | Pregunta |
|---|---|
| `pred_grupo_a` | `d.groupby("g")["x"].sum()["a"]` |
| `pred_n_grupos` | `len(d.groupby("g")["x"].sum())` |

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_1()

<details><summary>💡 Pista 1</summary>

Todos siguen el patrón `ventas.groupby("grupo")["valor"].funcion()`.
</details>

<details><summary>💡 Pista 2</summary>

Para `ventas_tienda_canal`, pasa una lista a `groupby`: `ventas.groupby(["tienda", "canal"])`. El redondeo va al final, con `.round(2)`.
</details>

---
## 2. Varias funciones a la vez: `agg`

### 📘 Concepto
`agg` aplica varias funciones en un solo paso:
- **Sobre una columna**, con una lista de nombres: `df.groupby("g")["x"].agg(["sum", "mean", "count"])`. Las columnas del resultado se llaman como las funciones.
- **Agregación con nombre**, para elegir el nombre de cada columna del resultado y resumir columnas distintas:

```python
df.groupby("g").agg(
    total=("monto", "sum"),
    maximo=("unidades", "max"),
)
```

Cada argumento es `nombre_nuevo=("columna", "función")`. `count` cuenta los valores no nulos; `nunique`, los valores distintos.

In [ ]:
pedidos_ej = pd.DataFrame({"vendedor": ["Ana", "Luis", "Ana", "Rosa", "Luis"],
                           "monto": [120.0, 80.0, 45.5, 300.0, 60.0],
                           "items": [2, 1, 1, 5, 3]})
print(pedidos_ej.groupby("vendedor")["monto"].agg(["sum", "mean", "count"]))
print(pedidos_ej.groupby("vendedor").agg(total=("monto", "sum"), items_max=("items", "max")))

### ✍️ Tu turno · Ejercicio 2: resúmenes con varias métricas
1. `resumen_tienda`: por tienda, con estas columnas en este orden: `total` (suma del importe), `promedio` (promedio del importe), `n_ventas` (cuántas ventas) y `max_unidades` (máximo de unidades en una venta).
2. `resumen_categoria`: por categoría, la suma, el promedio y el conteo del importe, usando `agg` con una lista de funciones.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_2()

<details><summary>💡 Pista 1</summary>

`resumen_tienda` usa agregación con nombre; `resumen_categoria`, una lista de funciones sobre la columna importe.
</details>

<details><summary>💡 Pista 2</summary>

`ventas.groupby("tienda").agg(total=("importe", "sum"), promedio=("importe", "mean"), ...)`: completa las otras dos con el mismo formato.
</details>

---
## 3. Contar categorías: `value_counts` y `crosstab`

### 📘 Concepto
- `df["col"].value_counts()` cuenta cuántas veces aparece cada valor, de mayor a menor. Con `normalize=True` devuelve **proporciones** (suman 1).
- `pd.crosstab(df["a"], df["b"])` cruza dos columnas: filas con los valores de `a`, columnas con los de `b` y, en cada celda, cuántas filas tienen esa combinación.
- Con `normalize="index"`, cada **fila** suma 1: responde "dentro de cada `a`, ¿qué proporción es de cada `b`?". Con `normalize="columns"`, suma 1 cada columna.

In [ ]:
pedidos_ej = pd.DataFrame({"vendedor": ["Ana", "Luis", "Ana", "Rosa", "Luis", "Ana"],
                           "canal": ["app", "tienda", "tienda", "app", "app", "app"]})
print(pedidos_ej["canal"].value_counts())
print(pedidos_ej["canal"].value_counts(normalize=True))
print(pd.crosstab(pedidos_ej["vendedor"], pedidos_ej["canal"]))
print(pd.crosstab(pedidos_ej["vendedor"], pedidos_ej["canal"], normalize="index").round(2))

### ✍️ Tu turno · Ejercicio 3: canales de venta
**Parte A.** Con `ventas`:
1. `conteo_canal`: cuántas ventas hay por canal.
2. `prop_canal`: la proporción de ventas de cada canal, redondeada a 3 decimales.
3. `tabla_cruzada`: cuántas ventas hay por tienda (filas) y canal (columnas).
4. `mezcla_canal`: la misma tabla, pero con la proporción de cada canal **dentro de cada tienda**, redondeada a 3 decimales.

**Parte B.** Predice **sin ejecutar**: `pred_suma_normalize` = `ventas["canal"].value_counts(normalize=True).sum()`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_3()

<details><summary>💡 Pista 1</summary>

`value_counts` se aplica a una columna; `crosstab` recibe dos columnas.
</details>

<details><summary>💡 Pista 2</summary>

Para `mezcla_canal`, cada fila (tienda) debe sumar 1: eso es `normalize="index"`.
</details>

---
## 4. Tablas dinámicas: `pivot_table`

### 📘 Concepto
`pivot_table` es la tabla dinámica de una hoja de cálculo:

```python
df.pivot_table(index="filas", columns="columnas", values="valor", aggfunc="sum", fill_value=0)
```

- `aggfunc` elige cómo resumir (`"sum"`, `"mean"`, `"count"`...).
- Donde una combinación no tiene datos, aparece `NaN`. Con `fill_value=0` se rellena con 0: tiene sentido para sumas (no hubo ventas) pero no para promedios (un promedio de 0 sería falso).

In [ ]:
pedidos_ej = pd.DataFrame({"vendedor": ["Ana", "Luis", "Ana", "Rosa", "Luis"],
                           "canal": ["app", "tienda", "tienda", "app", "app"],
                           "monto": [120.0, 80.0, 45.5, 300.0, 60.0]})
print(pedidos_ej.pivot_table(index="vendedor", columns="canal", values="monto", aggfunc="sum"))
print(pedidos_ej.pivot_table(index="vendedor", columns="canal", values="monto", aggfunc="sum", fill_value=0))

### ✍️ Tu turno · Ejercicio 4: tablas dinámicas de ventas
**Parte A.**
1. `pivote`: el importe total con tiendas en las filas y categorías en las columnas, con 0 donde no hubo ventas.
2. `pivote_unid`: el promedio de unidades por venta con categorías en las filas y canales en las columnas, redondeado a 2 decimales y **sin** rellenar los vacíos (¿qué combinación queda vacía y por qué?).

**Parte B.** Predice **sin ejecutar**: `pred_sin_fill` es lo que aparece en `pivote_unid` en la celda de una combinación sin ventas (un número o `"nan"`).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_4()

<details><summary>💡 Pista 1</summary>

Los dos siguen el patrón del ejemplo; cambia `index`, `columns`, `values` y `aggfunc`.
</details>

<details><summary>💡 Pista 2</summary>

`pivote` lleva `fill_value=0`; `pivote_unid` lleva `aggfunc="mean"` y `.round(2)` al final.
</details>

---
## 5. Tasas de conversión y ordenar resultados

### 📘 Concepto
Si una columna vale 1 cuando pasó algo (una compra) y 0 cuando no, **su promedio es la tasa**: la proporción de casos en que pasó. Por eso `df.groupby("segmento")["convirtio"].mean()` da la tasa de conversión de cada segmento.

Para ordenar:
- `s.sort_values(ascending=False)` ordena una Series; `df.sort_values(["a", "b"], ascending=[True, False])` ordena un DataFrame por varias columnas.
- `s.nlargest(n)` y `s.nsmallest(n)` devuelven los `n` mayores o menores, ya ordenados.
- `s.idxmax()` da la etiqueta del máximo.

In [ ]:
campana_ej = pd.DataFrame({"canal": ["email", "sms", "email", "sms", "email", "sms"],
                           "compro": [1, 0, 0, 0, 1, 1]})
tasas_ej = campana_ej.groupby("canal")["compro"].mean()
print(tasas_ej, tasas_ej.idxmax())
print(pd.Series([5, 9, 1, 7], index=["a", "b", "c", "d"]).nlargest(2))

### ✍️ Tu turno · Ejercicio 5: ¿qué funcionó en la campaña?
Con `contactos`:
1. `tasa_segmento`: la tasa de conversión de cada segmento, redondeada a 3 decimales, y `mejor_segmento`: la etiqueta del segmento con mayor tasa.
2. `tasa_canal_ordenada`: la tasa de conversión de cada canal de contacto, ordenada de mayor a menor (sin redondear).

Con `ventas`:

3. `top3_productos`: los 3 productos con mayor importe total, de mayor a menor.
4. `ventas_ordenadas`: `ventas` ordenado por tienda (de la A a la Z) y, dentro de cada tienda, de mayor a menor importe.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_ejercicio_5()

<details><summary>💡 Pista 1</summary>

La tasa es el promedio de `convirtio`. `nlargest` se aplica sobre el total por producto.
</details>

<details><summary>💡 Pista 2</summary>

Para `ventas_ordenadas`: `sort_values` con una lista de dos columnas y una lista de dos valores en `ascending`.
</details>

---
## 🏋️ Reto final: perfil de los clientes del banco
Con `movs` (sin modificarlo):
1. `egresos_cliente`: la suma de los montos **negativos** de cada cliente, con 2 decimales. (¿Aparecen todos los clientes?)
2. `resumen_segmento`: por segmento, con las columnas `n_clientes` (clientes distintos), `n_movs` (cantidad de movimientos) y `monto_medio` (monto promedio), redondeado a 2 decimales.
3. `uso_canal`: la proporción de movimientos de cada canal dentro de cada segmento, con 3 decimales.
4. `mes_tipo`: la suma del monto con meses en las filas y tipos de movimiento en las columnas, con 0 donde no hubo movimientos y 2 decimales.
5. `top5_depositantes`: los 5 clientes que más dinero depositaron (tipo `"deposito"`), de mayor a menor, con 2 decimales.
6. `pct_app_segmento`: el porcentaje de movimientos hechos por `"app"` en cada segmento, con 1 decimal. Pista: crea en una copia de `movs` una columna `es_app`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Los puntos 1 y 5 filtran filas antes de agrupar. Los puntos 3 y 4 son `crosstab` y `pivot_table`.
</details>

<details><summary>💡 Pista 2</summary>

Para el punto 6: `copia = movs.copy()`, `copia["es_app"] = copia["canal"] == "app"`, y luego el promedio de `es_app` por segmento, por 100 y redondeado.
</details>

---
## 🚀 Nivel pro (opcional)
1. `participacion`: para cada venta de `ventas`, qué parte del total de **su** tienda representa, con 4 decimales y en el orden original. Investiga `groupby(...)[...].transform("sum")`: devuelve el total del grupo repetido en cada fila.
2. `rank_en_tienda`: la posición de cada venta dentro de su tienda según el importe (1 para la mayor; los empates comparten el menor puesto). Investiga `groupby(...)[...].rank(ascending=False, method="min")`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Explicar los tres pasos de `groupby` y qué queda en el índice del resultado.
- [ ] Agrupar por una o por varias columnas.
- [ ] Usar `agg` con una lista de funciones y con agregación con nombre.
- [ ] Calcular proporciones con `value_counts(normalize=True)`.
- [ ] Explicar qué responde `crosstab` con `normalize="index"`.
- [ ] Construir una `pivot_table` y decidir cuándo usar `fill_value=0`.
- [ ] Calcular una tasa de conversión como el promedio de una columna de 0 y 1.
- [ ] Ordenar con `sort_values` y quedarme con los mayores con `nlargest`.

**Próxima sesión (S12):** combinar tablas con `merge` y `concat`, fechas con `to_datetime` y encadenar métodos.